# Can we predict next-month account churn?

**Objective.** Determine whether the fields available at a monthly account snapshot can predict `churn_next_month`, and estimate performance on genuinely later months.

**Short answer.** Yes, the snapshot-time fields contain useful **ranking** signal, but not enough for fully automated decisions. On the untouched Aug–Nov 2025 test period, the selected model reaches roughly 0.70 ROC AUC and doubles the precision of random targeting. Its raw probabilities are not well calibrated because churn prevalence changes substantially over time.

This notebook deliberately:

- excludes identifiers and fields unavailable at prediction time;
- uses chronological, walk-forward validation rather than a random row split;
- leaves the final four months untouched until the end;
- reports PR AUC, lift, calibration metrics, and a capacity-based top-decile policy;
- makes all results reproducible from `data.csv` and `columns.csv`.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, log_loss, brier_score_loss,
    precision_recall_curve, roc_curve
)

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 120)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_PATH = Path("data.csv")
DICTIONARY_PATH = Path("columns.csv")
assert DATA_PATH.exists() and DICTIONARY_PATH.exists(), "Run this notebook beside data.csv and columns.csv"

data = pd.read_csv(DATA_PATH)
dictionary = pd.read_csv(DICTIONARY_PATH)
data["snapshot_date"] = pd.to_datetime(data["snapshot_month"], format="%Y-%m")

print(f"Rows: {len(data):,} | accounts: {data.account_ref.nunique():,} | months: {data.snapshot_month.nunique()}")
display(dictionary)


## 1. Data quality and target behavior

Each row is an account-month snapshot. Multiple rows per account are expected, so a random row split would mix an account's earlier and later observations and would not represent forecasting future months. The deployment-like evaluation below splits on calendar time.


In [ ]:
quality = pd.Series({
    "rows": len(data),
    "columns": data.shape[1] - 1,  # exclude parsed helper date
    "accounts": data["account_ref"].nunique(),
    "months": data["snapshot_month"].nunique(),
    "missing_values": int(data.drop(columns="snapshot_date").isna().sum().sum()),
    "exact_duplicate_rows": int(data.drop(columns="snapshot_date").duplicated().sum()),
    "duplicate_account_months": int(data.duplicated(["account_ref", "snapshot_month"]).sum()),
    "overall_churn_rate": data["churn_next_month"].mean(),
})
display(quality.to_frame("value"))

assert set(data["churn_next_month"].unique()) <= {0, 1}
assert quality["missing_values"] == 0
assert quality["duplicate_account_months"] == 0


In [ ]:
monthly = (data.groupby("snapshot_date")["churn_next_month"]
           .agg(rows="size", churn_rate="mean").reset_index())

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly["snapshot_date"], monthly["churn_rate"], marker="o", linewidth=2)
ax.axvspan(pd.Timestamp("2025-08-01"), pd.Timestamp("2025-11-30"), color="#f59e0b", alpha=.18,
           label="final test period")
ax.axhline(data["churn_next_month"].mean(), color="gray", linestyle="--", label="overall mean")
ax.set(title="Churn prevalence changes sharply over time", xlabel="Snapshot month", ylabel="Next-month churn rate")
ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.legend()
plt.tight_layout()
plt.show()

display(monthly.assign(churn_rate=lambda x: x.churn_rate.map("{:.1%}".format)).tail(8))


The pronounced prevalence spike around late 2024 / early 2025 is a warning about **concept and prior-probability drift**. Ranking metrics can remain useful while raw probabilities become inaccurate. This is why the evaluation includes multiple historical forecasting folds and a later holdout.


## 2. Leakage audit

`lifetime_months` is described as total lifetime from a CRM export dated 2026-01-05. It was therefore computed **after** the snapshots and is unavailable at scoring time. In fact, it reveals the label exactly: when lifetime equals tenure, the snapshot is the account's final active month, so it cancels next month. It must be excluded.

`account_ref` is also excluded: it is an identifier, not a reusable predictor for new or future accounts. The remaining fields are plausible snapshot-time inputs.


In [ ]:
data["months_remaining_in_export"] = data["lifetime_months"] - data["tenure_months"]
leak_rule = (data["months_remaining_in_export"] == 0).astype(int)

leak_audit = pd.DataFrame({
    "check": ["Rule accuracy", "Rule ROC AUC", "Rows with impossible negative gap"],
    "value": [
        (leak_rule == data["churn_next_month"]).mean(),
        roc_auc_score(data["churn_next_month"], leak_rule),
        (data["months_remaining_in_export"] < 0).sum(),
    ],
})
display(leak_audit)
display(pd.crosstab(data["months_remaining_in_export"], data["churn_next_month"],
                    rownames=["lifetime - tenure"], colnames=["churn_next_month"]).head(8))


## 3. Evaluation design

The final test set is **Aug–Nov 2025** (3,362 snapshots) and remains untouched during model selection. Three earlier four-month forecasting folds compare:

- logistic regression versus a nonlinear histogram gradient booster;
- an expanding history versus the most recent 12 months.

The primary selection metric is **PR AUC**, appropriate for an imbalanced target. ROC AUC measures ranking across all thresholds; log loss and Brier score evaluate probability quality. No target-derived history is used.


In [ ]:
# Snapshot-time feature engineering only
data["calendar_month"] = data["snapshot_date"].dt.month.astype(str)

target = "churn_next_month"
categorical_features = ["plan", "calendar_month"]
numeric_features = ["seats", "monthly_fee", "tenure_months", "usage_hours", "support_tickets"]
features = categorical_features + numeric_features

excluded = ["account_ref", "snapshot_month", "snapshot_date", "lifetime_months",
            "months_remaining_in_export", target]
print("Model features:", features)
print("Explicitly excluded:", excluded)

test_mask = data["snapshot_month"].between("2025-08", "2025-11")
development = data.loc[~test_mask].copy()
test = data.loc[test_mask].copy()

folds = [
    {"train_end": "2024-07", "valid_start": "2024-08", "valid_end": "2024-11"},
    {"train_end": "2024-11", "valid_start": "2024-12", "valid_end": "2025-03"},
    {"train_end": "2025-03", "valid_start": "2025-04", "valid_end": "2025-07"},
]

split_summary = pd.DataFrame([
    {"split": "development", "months": "2024-01 to 2025-07", "rows": len(development),
     "churn_rate": development[target].mean()},
    {"split": "final test", "months": "2025-08 to 2025-11", "rows": len(test),
     "churn_rate": test[target].mean()},
])
display(split_summary.style.format({"churn_rate": "{:.1%}"}))


In [ ]:
def build_model(model_name):
    if model_name == "logistic":
        prep = ColumnTransformer([
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("numeric", StandardScaler(), numeric_features),
        ])
        estimator = LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)
    elif model_name == "hist_gradient_boosting":
        prep = ColumnTransformer([
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
            ("numeric", "passthrough", numeric_features),
        ])
        estimator = HistGradientBoostingClassifier(
            max_iter=150, learning_rate=.05, max_leaf_nodes=15,
            l2_regularization=1, random_state=RANDOM_STATE
        )
    else:
        raise ValueError(model_name)
    return Pipeline([("preprocess", prep), ("model", estimator)])


def metric_row(y, score):
    return {
        "pr_auc": average_precision_score(y, score),
        "roc_auc": roc_auc_score(y, score),
        "log_loss": log_loss(y, score),
        "brier": brier_score_loss(y, score),
    }


def training_start(train_end, window):
    if window == "expanding":
        return "2024-01"
    return str(pd.Period(train_end, freq="M") - (int(window) - 1))


candidates = [
    ("logistic", "expanding"),
    ("logistic", 12),
    ("hist_gradient_boosting", "expanding"),
    ("hist_gradient_boosting", 12),
]

cv_rows = []
for model_name, window in candidates:
    for fold_id, fold in enumerate(folds, start=1):
        start = training_start(fold["train_end"], window)
        train_fold = data[data["snapshot_month"].between(start, fold["train_end"])]
        valid_fold = data[data["snapshot_month"].between(fold["valid_start"], fold["valid_end"])]
        model = build_model(model_name)
        model.fit(train_fold[features], train_fold[target])
        score = model.predict_proba(valid_fold[features])[:, 1]
        cv_rows.append({
            "model": model_name, "window": str(window), "fold": fold_id,
            "train_months": f"{start}–{fold['train_end']}",
            "valid_months": f"{fold['valid_start']}–{fold['valid_end']}",
            "valid_rate": valid_fold[target].mean(), **metric_row(valid_fold[target], score)
        })

cv_results = pd.DataFrame(cv_rows)
cv_summary = (cv_results.groupby(["model", "window"], as_index=False)
              .agg(mean_pr_auc=("pr_auc", "mean"), mean_roc_auc=("roc_auc", "mean"),
                   mean_log_loss=("log_loss", "mean"), mean_brier=("brier", "mean"))
              .sort_values("mean_pr_auc", ascending=False))
display(cv_summary.style.format({c: "{:.3f}" for c in cv_summary.columns if c.startswith("mean_")}))


Logistic regression with the most recent 12 months has the best mean walk-forward PR AUC, while remaining interpretable. Its advantage over expanding-history logistic regression is small, but the rolling window is also a sensible response to the observed drift. This choice is fixed before opening the final test set.


## 4. Final untouched test

The selected model trains on Aug 2024–Jul 2025 (the 12 months immediately before the test). The baseline predicts the training-set churn rate for every row. For operational targeting, the top-decile policy flags exactly the highest-scored 10% of accounts; this avoids pretending there is a universal probability threshold while calibration is unstable.


In [ ]:
final_train = data[data["snapshot_month"].between("2024-08", "2025-07")].copy()
final_model = build_model("logistic")
final_model.fit(final_train[features], final_train[target])

y_test = test[target].to_numpy()
test_score = final_model.predict_proba(test[features])[:, 1]
baseline_score = np.full(len(test), final_train[target].mean())

results = pd.DataFrame([
    {"model": "constant training-rate baseline", **metric_row(y_test, baseline_score)},
    {"model": "selected logistic model", **metric_row(y_test, test_score)},
])
display(results.style.format({"pr_auc": "{:.3f}", "roc_auc": "{:.3f}",
                              "log_loss": "{:.3f}", "brier": "{:.3f}"}))

n_flagged = int(np.ceil(.10 * len(test)))
ranked_positions = np.argsort(-test_score)
flagged = ranked_positions[:n_flagged]
top_decile_precision = y_test[flagged].mean()
top_decile_recall = y_test[flagged].sum() / y_test.sum()
test_rate = y_test.mean()
lift = top_decile_precision / test_rate

policy = pd.Series({
    "test rows": len(test),
    "actual test churn rate": test_rate,
    "mean predicted probability": test_score.mean(),
    "accounts flagged (top 10%)": n_flagged,
    "precision among flagged": top_decile_precision,
    "share of all churners captured": top_decile_recall,
    "lift vs random targeting": lift,
})
display(policy.to_frame("value").style.format({"value": lambda x: f"{x:.3f}" if isinstance(x, float) else str(x)}))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

fpr, tpr, _ = roc_curve(y_test, test_score)
axes[0].plot(fpr, tpr, linewidth=2, label=f"Model (AUC={roc_auc_score(y_test, test_score):.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Random")
axes[0].set(xlabel="False-positive rate", ylabel="True-positive rate", title="ROC curve")
axes[0].legend()

precision, recall, _ = precision_recall_curve(y_test, test_score)
axes[1].plot(recall, precision, linewidth=2, label=f"Model (AP={average_precision_score(y_test, test_score):.3f})")
axes[1].axhline(test_rate, linestyle="--", color="gray", label=f"Random baseline ({test_rate:.3f})")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision–recall curve")
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. What drives the model?

Coefficients below are associations, not causal effects. Numeric features were standardized, so their coefficient magnitudes are comparable to one another. One-hot category coefficients are relative to the regularized model intercept and other categories; they should not be interpreted in isolation as causal plan effects.


In [ ]:
feature_names = final_model.named_steps["preprocess"].get_feature_names_out()
coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": final_model.named_steps["model"].coef_[0],
})
coefficients["abs_coefficient"] = coefficients["coefficient"].abs()

display(coefficients.nlargest(8, "coefficient")[["feature", "coefficient"]].style.format({"coefficient": "{:+.3f}"}))
display(coefficients.nsmallest(8, "coefficient")[["feature", "coefficient"]].style.format({"coefficient": "{:+.3f}"}))


## 6. Conclusion and recommendation


In [ ]:
display(Markdown(f''' 
### Yes—for prioritization, with guardrails

On the untouched **Aug–Nov 2025** test period:

- **ROC AUC:** {roc_auc_score(y_test, test_score):.3f}
- **PR AUC:** {average_precision_score(y_test, test_score):.3f}, versus a random-ranking baseline of {test_rate:.3f}
- **Top 10% precision:** {top_decile_precision:.1%}; this captures {top_decile_recall:.1%} of all churners and gives **{lift:.2f}× lift** over random targeting
- **Calibration warning:** mean predicted risk is {test_score.mean():.1%}, while actual churn is {test_rate:.1%}

This is useful for ranking accounts into a human-led retention queue or for a randomized retention experiment. It is **not** accurate enough to treat scores as stable probabilities or to trigger costly actions without measuring business value.

Recommended next steps:

1. Deploy as a ranking/pilot and choose outreach capacity from expected value, not a default 0.5 threshold.
2. Retrain monthly on a recent window and recalibrate on the newest completed months; monitor PR AUC, lift, and calibration by month and plan.
3. Investigate the late-2024 churn spike (pricing, migration, contract cycle, or data-generation change).
4. Add only prediction-time features such as usage trend, payment failures, engagement breadth, seat utilization, renewal date, and recent support sentiment.
5. Keep `lifetime_months` permanently out of training—it encodes the future and produces a fake perfect result.
6. Validate intervention impact with a randomized holdout; prediction alone does not show which accounts are persuadable.
'''))
